In [0]:
use catalog identifier(:catalog)

In [0]:
CREATE OR REPLACE TABLE gold.fact_sales AS
SELECT
  l.l_orderkey,
  l.l_linenumber,
  o.o_orderdate,
  o.o_orderpriority,
  o.o_custkey,
  l.l_partkey,
  l.l_suppkey,
  l.l_quantity,
  l.l_extendedprice,
  l.l_discount,
  l.l_tax,
  l.l_extendedprice * (1 - l.l_discount) AS net_revenue,
  l.l_returnflag,
  l.l_linestatus,
  l.l_shipdate,
  l.l_receiptdate,
  l.l_shipmode,
  o.o_totalprice,
  c.c_name                AS customer_name,
  c.c_mktsegment,
  nc.n_name               AS customer_nation,
  rc.r_name               AS customer_region,
  p.p_name                AS product_name,
  p.p_type                AS product_type,
  p.p_mfgr                AS product_mfgr,
  p.p_brand               AS product_brand,
  s.s_name                AS supplier_name,
  ns.n_name               AS supplier_nation,
  rs.r_name               AS supplier_region,
  o.o_orderstatus
FROM silver.lineitem l
INNER JOIN silver.orders   o  ON o.o_orderkey  = l.l_orderkey
INNER JOIN silver.customer c  ON c.c_custkey   = o.o_custkey
INNER JOIN silver.nation   nc ON nc.n_nationkey = c.c_nationkey
INNER JOIN silver.region   rc ON rc.r_regionkey = nc.n_regionkey
INNER JOIN silver.part     p  ON p.p_partkey   = l.l_partkey
INNER JOIN silver.supplier s  ON s.s_suppkey   = l.l_suppkey
INNER JOIN silver.nation   ns ON ns.n_nationkey = s.s_nationkey
INNER JOIN silver.region   rs ON rs.r_regionkey = ns.n_regionkey

In [0]:
CREATE OR REPLACE TABLE gold.dim_customer AS
select c.c_custkey,
       c.c_name,
       c.c_address,
       c.c_nationkey,
       c.c_phone,
       c.c_acctbal,
       c.c_mktsegment,
       c.c_comment,
       n.n_nationkey,
       n.n_name,
       n.n_regionkey,
       n.n_comment,
       r.r_regionkey,
       r.r_name,
       r.r_comment
       
from  silver.customer c 
 INNER join silver.nation n on n.n_nationkey=c.c_nationkey
 INNER join silver.region r on r.r_regionkey=n.n_regionkey      





In [0]:
CREATE OR REPLACE TABLE gold.dim_product AS
SELECT
  p.p_partkey,
  p.p_name,
  p.p_mfgr,
  p.p_brand,
  p.p_type,
  p.p_size,
  p.p_container,
  p.p_retailprice,
  p.p_comment,
  SUM(ps.ps_availqty)     AS total_availqty,
  AVG(ps.ps_supplycost)   AS avg_supplycost,
  COUNT(DISTINCT ps.ps_suppkey) AS supplier_count
FROM silver.part p
LEFT JOIN silver.partsupp ps ON ps.ps_partkey = p.p_partkey
GROUP BY p.p_partkey, p.p_name, p.p_mfgr, p.p_brand, p.p_type,
         p.p_size, p.p_container, p.p_retailprice, p.p_comment


In [0]:

CREATE OR REPLACE TABLE gold.dim_supplier AS
select s.s_suppkey,s.s_name,s.s_address,s.s_nationkey,s.s_phone,s.s_acctbal,s.s_comment,n.n_name,n.n_regionkey,n.n_comment,r.r_name,r.r_comment
FROM silver.supplier s
INNER JOIN silver.nation n ON n.n_nationkey = s.s_nationkey
INNER JOIN silver.region r ON r.r_regionkey=n.n_regionkey
   

In [0]:
CREATE OR REPLACE TABLE gold.dim_geography AS
SELECT
  n.n_nationkey,
  n.n_name,
  n.n_regionkey,
  n.n_comment,
  r.r_name,
  r.r_comment
FROM silver.nation n
INNER JOIN silver.region r ON r.r_regionkey = n.n_regionkey

In [0]:

CREATE OR REPLACE VIEW gold.vw_annual_sales_kpis AS
SELECT
  YEAR(o_orderdate) AS year,
  SUM(net_revenue) AS net_revenue,
  COUNT(DISTINCT l_orderkey) AS orders,
  COUNT(DISTINCT o_custkey) AS active_customers,
  CAST(SUM(net_revenue) / NULLIF(COUNT(DISTINCT l_orderkey), 0) AS DECIMAL(18,2)) AS avg_order_value
FROM gold.fact_sales
GROUP BY YEAR(o_orderdate);
